# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tushar-sharma001/Flyrank-Ml-Internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os

if not os.path.exists("Flyrank-Ml-Internship"):
    !git clone https://github.com/tushar-sharma001/Flyrank-Ml-Internship.git

os.chdir("Flyrank-Ml-Internship")
print(os.getcwd())
!ls data/raw/

Cloning into 'Flyrank-Ml-Internship'...
remote: Enumerating objects: 154, done.
remote: Counting objects: 100% (154/154), done.
remote: Compressing objects: 100% (111/111), done.
remote: Total 154 (delta 62), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (154/154), 1.89 MiB | 9.43 MiB/s, done.
Resolving deltas: 100% (62/62), done.
/content/Flyrank-Ml-Internship
content_refresh_anonymized.csv


In [3]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

MIN_N = 50
MIN_N_CROSS = 30

def spearman_verdict(x, y, label, n_floor=MIN_N):
    mask = x.notna() & y.notna()
    n = mask.sum()
    if n < n_floor:
        return f"{label}: INSUFFICIENT DATA (n={n})"
    rho, p = stats.spearmanr(x[mask], y[mask])
    if p >= 0.05:
        verdict = "MIXED" if abs(rho) < 0.1 else "FALSE"
    else:
        verdict = "CONFIRMED" if rho > 0 else "OPPOSITE"
    return f"{label}: rho={rho:.3f}, p={p:.4f}, n={n} -> {verdict}"

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

Before testing any signal, I looked at the shape of the key fields I'll use below: word_count,
clicks/impressions, engagement_rate, scroll_rate, and avg_position.

- Rate columns (ctr, engagement_rate, scroll_rate, ai_traffic_pct, trend_pct) are already
  expressed as ×100 percentages in this dataset, not fractions — I did not re-scale them.
- avg_position has 1,205 rows at exactly 0, which the data dictionary flags as "no data," not
  rank zero. I excluded those before looking at the real distribution.
- word_count and click/impression counts are heavily right-skewed, as expected for web traffic
  data — a small number of pages account for a disproportionate share of volume.
- scroll_rate and ai_traffic_pct can legitimately exceed 100 because their numerator and
  denominator come from different measurement systems — not a data error.

**Decision for the rest of this notebook:** I'll use log1p on traffic-count columns and
Spearman (rank) correlation everywhere else, since Pearson correlation on raw heavy-tailed
values is dominated by outliers and isn't reliable here.

In [4]:
rate_cols = ["ctr", "engagement_rate", "scroll_rate", "ai_traffic_pct", "trend_pct"]
print(df[rate_cols].describe())

pos = df.loc[df["avg_position"] > 0, "avg_position"]
print(f"avg_position: {len(df) - len(pos)} rows are 'no data' (excluded)")
print(pos.describe())

for col in ["word_count", "clicks_90d", "impressions_90d", "sessions_90d"]:
    print(col, "skew:", df[col].skew())

df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])

                ctr  engagement_rate   scroll_rate  ai_traffic_pct  \
count  30000.000000     30000.000000  29875.000000    30000.000000   
mean       0.510733         2.534520     18.212921        0.768196   
std        3.279162         8.310096     29.472768        7.429454   
min        0.000000         0.000000      0.000000        0.000000   
25%        0.000000         0.000000      0.000000        0.000000   
50%        0.070000         0.000000      5.000000        0.000000   
75%        0.290000         1.350000     23.530000        0.000000   
max      100.000000       100.000000    300.000000      300.000000   

          trend_pct  
count  26612.000000  
mean      -4.785969  
std      473.861780  
min     -100.000000  
25%      -62.600000  
50%      -33.500000  
75%        0.000000  
max    44900.000000  
avg_position: 1205 rows are 'no data' (excluded)
count    28795.000000
mean        17.026268
std         15.152439
min          0.100000
25%          6.700000
50%         

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Three commonly-believed claims, each tested against the data with a verdict:
CONFIRMED / OPPOSITE / MIXED / FALSE / INSUFFICIENT DATA (below the n=50 floor).

1. **"Longer content gets more traffic."** word_count vs log(clicks), Spearman.
2. **"Higher engagement predicts a better ranking position."** engagement_rate vs
   avg_position (excluding the 0 = no-data rows), Spearman.
3. **"scroll_rate varies meaningfully by content_type."** Grouped medians with visible n
   per group — no verdict issued for any group under the sample-size floor.

[Fill in actual verdicts here once the cell below runs — e.g.:]
- Signal 1: **[VERDICT]** — rho=[x], n=[x]
- Signal 2: **[VERDICT]** — rho=[x], n=[x]
- Signal 3: **[VERDICT]** — content_type [X] has notably higher/lower median scroll_rate;
  [type Y] didn't clear the sample floor.

In [5]:
results = []

# Signal 1: longer content -> more clicks
results.append(spearman_verdict(df["word_count"], df["log_clicks_90d"], "word_count vs log_clicks_90d"))

# Signal 2: higher engagement_rate -> better (lower) avg_position
valid_pos = df["avg_position"].replace(0, np.nan)
results.append(spearman_verdict(df["engagement_rate"], valid_pos, "engagement_rate vs avg_position"))

# Signal 3: scroll_rate by content_type — grouped medians with visible n
grp = df.groupby("content_type")["scroll_rate"].agg(median="median", n="count")
grp["verdict_ok"] = grp["n"] >= MIN_N
print(grp)

for r in results:
    print(r)

                    median      n  verdict_ok
content_type                                 
comparison article   50.00    695        True
feedly article       20.83   2095        True
keyword article       4.17  27085        True
word_count vs log_clicks_90d: rho=0.204, p=0.0000, n=22301 -> CONFIRMED
engagement_rate vs avg_position: rho=-0.089, p=0.0000, n=28795 -> OPPOSITE


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

FlyRank's `recommended_action == "refresh"` flag implicitly assumes that flagged pages show
measurably lower engagement than non-flagged pages — that's the signal I'm auditing here.

Important constraint: `trend_direction` and `trend_pct` are never used as inputs — they're
downstream of the same logic that produces the label, so using them would just test the rule
against itself. I recompute a decline signal independently from raw engagement_rate instead.

I also checked missingness by content_type first, since a blind fillna(0) upstream could have
quietly baked a category effect into the flag's numbers rather than a genuine decline signal.

[Fill in after running:]
- Missingness pattern: [content_type X] has notably higher missing engagement_rate.
- Overall verdict: **[CONFIRMED / MIXED / FALSE]** — flagged median=[x] (n=[x]) vs
  not-flagged median=[x] (n=[x]), p=[x].
- Per-content-type robustness check: the effect [held up / did not hold up] consistently across
  types; [type Y] had insufficient data either way.

In [7]:
# Real flag: trend_direction. Assumption to audit: pages flagged "declining" should show
# meaningfully lower engagement_rate than non-declining pages.
# NOTE: trend_direction/trend_pct are never used as model FEATURES per the data dictionary's
# label trap — but auditing their own assumption here is exactly what this notebook is for.

print(df["trend_direction"].value_counts(dropna=False))

DECLINE_VALUE = "down"   # confirm this matches the printed value_counts above exactly
assert DECLINE_VALUE in df["trend_direction"].astype(str).unique(), \
    "check exact spelling/case from the value_counts printed above"

flagged = df[df["trend_direction"] == DECLINE_VALUE].copy()
not_flagged = df[df["trend_direction"] != DECLINE_VALUE].copy()
print(f"n declining={len(flagged)}, n not_declining={len(not_flagged)}")

miss_by_type = df.groupby("content_type")["engagement_rate"].apply(lambda s: s.isna().mean())
print("Missing engagement_rate rate by content_type:\n", miss_by_type)

eng_flagged = flagged["engagement_rate"].dropna()
eng_not = not_flagged["engagement_rate"].dropna()

if len(eng_flagged) >= MIN_N and len(eng_not) >= MIN_N:
    stat, p = stats.mannwhitneyu(eng_flagged, eng_not, alternative="two-sided")
    verdict = "CONFIRMED" if (p < 0.05 and eng_flagged.median() < eng_not.median()) else "MIXED/FALSE"
    print(f"Declining median engagement={eng_flagged.median():.3f} (n={len(eng_flagged)}), "
          f"not-declining median={eng_not.median():.3f} (n={len(eng_not)}), p={p:.4f} -> {verdict}")
else:
    print("INSUFFICIENT DATA for this verdict")

# Robustness check: rerun per content_type
for ctype in df["content_type"].dropna().unique():
    sub = df[df["content_type"] == ctype]
    f = sub.loc[sub["trend_direction"] == DECLINE_VALUE, "engagement_rate"].dropna()
    nf = sub.loc[sub["trend_direction"] != DECLINE_VALUE, "engagement_rate"].dropna()
    if len(f) >= MIN_N_CROSS and len(nf) >= MIN_N_CROSS:
        _, p = stats.mannwhitneyu(f, nf, alternative="two-sided")
        print(f"  [{ctype}] n_decline={len(f)}, n_other={len(nf)}, p={p:.4f}")
    else:
        print(f"  [{ctype}] insufficient data (n_decline={len(f)}, n_other={len(nf)})")

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64
n declining=16262, n not_declining=13738
Missing engagement_rate rate by content_type:
 content_type
comparison article    0.0
feedly article        0.0
keyword article       0.0
Name: engagement_rate, dtype: float64
Declining median engagement=0.000 (n=16262), not-declining median=0.000 (n=13738), p=0.1100 -> MIXED/FALSE
  [keyword article] n_decline=15262, n_other=11945, p=0.0000
  [feedly article] n_decline=601, n_other=1495, p=0.0000
  [comparison article] n_decline=399, n_other=298, p=0.6388


In [8]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'log_clicks_90d', 'log_impressions_90d']


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*


The "declining" trend flag's underlying assumption — that flagged pages show measurably lower engagement_rate — is [CONFIRMED/MIXED/FALSE] overall, and [held up/broke down] when checked per content type. Content teams should treat trend_direction as a directional starting point for triage, not a standalone signal of content quality — especially for [weak content type], where this sample doesn't clearly support the flag's assumption.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.